# Your first agent, then a different harness

In this walkthrough, you'll ask an agent to draft a customer reply. Then you'll
switch from **DeepAgents to Pydantic AI** and run the same code with the same model.

You need an **OpenAI API key with API credit**. A ChatGPT subscription does not
include API credit. Calls in this notebook are billed to your OpenAI account.

**How to use this notebook:** click the ▶ button to the left of each code cell,
starting at the top. Wait for it to finish before moving to the next step.
Colab may ask you to connect a runtime or confirm running this notebook; the
standard CPU runtime is enough. Nothing needs to be installed on your laptop.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/00_agent.ipynb)

## 1. Install LiteAgents

Click ▶ on the next cell. It installs LiteAgents and the two harnesses we'll try.
This may take about a minute. Wait until you see **Installed. Continue to step 2.**

This preview is installed from our GitHub release. You don't need to clone a repository.

In [ ]:
%pip install -q "liteagents[deepagents,pydantic-ai] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a3/liteagents-0.3.0a3-py3-none-any.whl" -c https://github.com/BerriAI/liteagents/releases/download/v0.3.0a3/constraints-tested.txt

print("Installed. Continue to step 2.")

## 2. Add your OpenAI API key

Get a key from [OpenAI's API keys page](https://platform.openai.com/api-keys).
Run the cell below, paste your key into the box **under the cell**, and press Enter.
The input is hidden; you do not need to configure Colab Secrets.

When you see **Key set. Continue to step 3.**, you're ready. The key is kept in
this runtime's environment, not in the notebook source. Rerunning this cell
reuses the key you've already entered.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY") or getpass("Paste your OpenAI API key: ")
if not os.environ["OPENAI_API_KEY"].strip():
    raise ValueError("No key entered. Run this cell again and paste your OpenAI API key.")
print("Key set. Continue to step 3.")

## 3. Run your first agent

Run this cell as it is. It asks **DeepAgents** to use an **OpenAI model** to write
a customer reply. You should see a short reply printed below the code.

Two settings matter here:
- `harness` chooses the framework that runs the agent.
- `model` chooses the model it calls. LiteLLM connects to OpenAI using your key.

`query()` is the SDK call. The small `ask()` function prints the text from its
messages so we can reuse exactly the same application code in the next step.

In [ ]:
from liteagents import AssistantMessage, LiteAgentOptions, ProfileOptions, TextBlock, query

profile = ProfileOptions(
    harness="deepagents",
    model="openai/gpt-5.4-mini",
    tools=[],
)

async def ask(prompt):
    async for message in query(prompt=prompt, options=LiteAgentOptions(profile=profile)):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)

prompt = "Write a friendly two-sentence reply to a customer whose package will arrive tomorrow."
await ask(prompt)

You should get something like: *“Thanks for your patience! Your package is on
its way and is expected to arrive tomorrow.”* The exact wording will vary.

That's a working agent: your prompt went through DeepAgents to OpenAI and came
back as LiteAgents messages. `tools=[]` keeps this first example focused on text.

## 4. Switch the harness — keep everything else

Run the next cell. The first line changes only the harness to **Pydantic AI**.
The second calls the same `ask()` function with the same prompt, OpenAI model,
and API key. Both harnesses were installed in step 1, so there's no more setup.

In [ ]:
profile.harness = "pydantic-ai"
await ask(prompt)

You'll see another customer reply. Its wording may differ because this is a
new run, but the way your application sends prompts and reads responses is unchanged.
That's what switching harnesses means in LiteAgents.

## 5. Make it yours

Edit the question inside the quotes below, then click ▶. You can rerun this cell
as many times as you like. Each call starts a fresh conversation and uses the
currently selected harness, Pydantic AI.

In [ ]:
await ask("Explain what an agent harness does in one sentence.")

To switch back, run `profile.harness = "deepagents"` before your next `ask()` call.
To keep your edits, choose **File → Save a copy in Drive**.

## Want to use Anthropic or OpenRouter?

The walkthrough above uses OpenAI to keep the first run simple. You can use a
different provider with either harness. Run **one** of these snippets in a new
code cell (click **+ Code** in the toolbar), using that provider's API key:

**Anthropic**
```python
os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
profile.model = "anthropic/claude-sonnet-4-6"
await ask(prompt)
```

**OpenRouter**
```python
os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
profile.model = "openrouter/anthropic/claude-sonnet-4.6"
await ask(prompt)
```

If you only have an Anthropic or OpenRouter key, use its key line in step 2 and
its model name in step 3 instead of the OpenAI defaults.
[More providers, including Bedrock](https://github.com/BerriAI/liteagents/blob/main/docs/models.md).
A LiteLLM gateway is optional; none of these examples needs one.

## Next, give your agent something to do

- [Add a Python tool](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/08_application_tools.ipynb): let the agent look up an order.
- [Connect tools through MCP and try more harnesses](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/10_harness_switch.ipynb): use the same tools across frameworks.
- [Continue a conversation](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/01_quickstart.ipynb): keep history for follow-up questions.

## If a step fails

- **No API key / authentication error:** check the key and rerun step 2. OpenAI API billing is separate from ChatGPT.
- **Quota or model access error:** check your provider account's credit and model access. You can change `model` in step 3 to one your account supports.
- **`profile` or `ask` is not defined:** run step 3 first. Colab runs cells in the order you click them.
- **Colab asks for a runtime restart:** restart, then run steps 1–3 again. A fresh runtime needs installation and your key again.